# 법률 문서 검토 보조 - Colab 실행 데모

이 노트북은 `privacy-consent-review-ai` 프로젝트의 최신 분석 기능을 Google Colab에서 재현하기 위한 제출용 파일입니다.

현재 프로젝트는 **분류 모델 3개, 계약 필드 후보 모델 1개, BIO 필드 span fallback 모델 1개, Transformer NER 선택 모델 1개** 및 공식 법령 기반 규칙 엔진을 결합한 하이브리드 분석기입니다. 학습 과정은 `ai_model_training.ipynb`에서 재현할 수 있습니다.

분석 결과는 법 위반이나 계약 효력을 판정하지 않으며, 누락 가능성과 추가 검토 지점을 제시합니다.

## 1. 저장소와 실행 환경 준비

Colab에서는 함께 제공되는 `privacy-consent-review-ai-colab.zip`을 업로드하면 아직 GitHub에 푸시하지 않은 최신 작업본도 실행할 수 있습니다. ZIP을 업로드하지 않으면 GitHub 저장소를 복제합니다.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/sam3319/privacy-consent-review-ai.git"
PROJECT_DIR = Path("/content/privacy-consent-review-ai")

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if not IS_COLAB and Path("src").exists() and Path("data").exists():
    PROJECT_DIR = Path.cwd()
elif IS_COLAB and not (PROJECT_DIR / "src" / "employment_calculator.py").exists():
    from google.colab import files

    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
        print("불완전한 이전 프로젝트 폴더를 삭제했습니다.")

    print("최신 작업본을 사용하려면 privacy-consent-review-ai-colab.zip을 업로드하세요.")
    print("취소하거나 다른 파일을 선택하면 GitHub 저장소를 사용합니다.")
    uploaded_project = files.upload()
    zip_names = [name for name in uploaded_project if name.lower().endswith(".zip")]
    if zip_names:
        archive_path = Path("/content") / zip_names[0]
        archive_path.write_bytes(uploaded_project[zip_names[0]])
        shutil.unpack_archive(archive_path, Path("/content"))

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)],
        check=True,
    )

if IS_COLAB:
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "poppler-utils", "tesseract-ocr", "tesseract-ocr-kor"],
        check=True,
    )

os.chdir(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

project_path = str(PROJECT_DIR.resolve())
if project_path in sys.path:
    sys.path.remove(project_path)
sys.path.insert(0, project_path)

# 이전 실행에서 불러온 다른 src 패키지 캐시를 제거합니다.
import importlib
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()

required_module = PROJECT_DIR / "src" / "employment_calculator.py"
if not required_module.is_file():
    raise FileNotFoundError(
        f"필수 모듈이 없습니다: {required_module}. 최신 ZIP을 다시 업로드하세요."
    )

print(f"프로젝트 경로: {PROJECT_DIR.resolve()}")
print(f"근로 계산 모듈: {required_module.resolve()}")
print("환경 준비 완료")

## 2. 분석 함수 불러오기

In [ ]:
from pprint import pprint
import importlib
import sys

# 노트북 재실행 시 남아 있는 src 패키지 캐시를 제거합니다.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()

from src.analyzer import analyze_document
from src.contract_analyzer import analyze_contract
from src.document_classifier import detect_document_type_hybrid
from src.document_io import extract_text
from src.employment_calculator import calculate_wage_estimate
from src.legal_retrieval import search_legal_basis
from src.model_security import model_integrity_status
from src.special_contract_analyzer import (
    analyze_employment_contract,
    analyze_housing_contract,
)

def summarize_result(result):
    print(f"문서 유형: {result['document_type']}")
    print(f"핵심정보 탐지율: {result['completeness']}%")
    print("\n[구조화 추출]")
    for field in result.get("extracted_fields", []):
        value = field["value"] or "찾지 못함"
        print(f"- {field['label']}: {value} ({field['status']})")
    print("\n[검토 신호]")
    if not result["findings"]:
        print("- 현재 규칙으로 탐지된 검토 신호가 없습니다.")
    for finding in result["findings"]:
        print(
            f"- [{finding['severity']}] {finding['title']} | "
            f"{finding['article']} | {finding['status']}"
        )

integrity = model_integrity_status()
print("분석 함수 로드 완료")
print(f"필수 모델 무결성: {'정상' if integrity['valid'] else '오류'}")
for model_status in integrity['models']:
    optional = "선택" if model_status.get('optional') else "필수"
    state = "정상" if model_status['valid'] else "fallback" if model_status.get('optional') else "오류"
    print(f"- {model_status['name']} ({optional}): {state}")
for warning in integrity.get('warnings', []):
    print(f"  참고: {warning}")

## 3. 문서 유형 자동 분류 확인

In [ ]:
sample_files = [
    "samples/complete_collection_consent.txt",
    "samples/risky_standard_terms_contract.txt",
    "samples/risky_housing_lease.txt",
    "samples/risky_employment_contract.txt",
]

for filename in sample_files:
    text = Path(filename).read_text(encoding="utf-8")
    detection = detect_document_type_hybrid(text)
    print(
        f"{Path(filename).name}: {detection['label']} "
        f"(신뢰도 {detection['confidence']}%)"
    )

## 4. 개인정보 수집·이용 동의서 분석

In [ ]:
privacy_text = Path("samples/complete_collection_consent.txt").read_text(
    encoding="utf-8"
)
privacy_result = analyze_document(privacy_text, "collection")
summarize_result(privacy_result)

## 5. 일반 약관형 계약서 분석

약관법 관련 검토 신호와 계약 핵심정보, 조항 구조, 당사자 관점 요약을 확인합니다.

In [ ]:
contract_text = Path("samples/risky_standard_terms_contract.txt").read_text(
    encoding="utf-8"
)
contract_result = analyze_contract(contract_text, perspective="을")
summarize_result(contract_result)

print("\n[을 관점 요약]")
pprint(contract_result["perspective_summary"])

## 6. 주택 임대차·전세·월세 계약서 분석

주택임대차보호법에 연결된 계약기간, 보증금 반환, 갱신요구권, 차임 증액 등의 검토 신호를 확인합니다.

In [ ]:
housing_text = Path("samples/risky_housing_lease.txt").read_text(
    encoding="utf-8"
)
housing_result = analyze_housing_contract(housing_text, perspective="임차인")
summarize_result(housing_result)

print("\n[보증금 담보여력 참고 계산]")
housing_with_risk = analyze_housing_contract(
    housing_text,
    perspective="임차인",
    financial_inputs={
        "property_value": 300_000_000,
        "senior_claims": 120_000_000,
        "senior_deposits": 30_000_000,
    },
)
pprint(housing_with_risk["deposit_risk"])

## 7. 근로계약서 분석

근로조건 서면 명시, 근로시간, 위약금, 임금 지급 방식 및 2026년 적용 최저임금 시간급 10,320원 미달 가능성을 확인합니다.

In [ ]:
employment_text = Path("samples/risky_employment_contract.txt").read_text(
    encoding="utf-8"
)
employment_result = analyze_employment_contract(
    employment_text,
    perspective="근로자",
)
summarize_result(employment_result)

print("\n[근로시간·예상 수당 참고 계산]")
pprint(
    calculate_wage_estimate(
        hourly_wage=12_000,
        regular_hours=40,
        overtime_hours=5,
        night_hours=2,
        holiday_hours=10,
    )
)

## 8. 사용자 문서 업로드 및 자동 분석

Colab에서 TXT, PDF, DOCX, PNG, JPG 파일을 업로드하면 텍스트 또는 OCR을 추출하고 문서 유형을 자동 감지하여 적절한 분석기를 실행합니다.

In [ ]:
def analyze_text_automatically(text):
    detection = detect_document_type_hybrid(text)
    document_type = detection["document_type"]

    if document_type == "standard_terms_contract":
        result = analyze_contract(text, perspective="을")
    elif document_type == "housing_lease":
        result = analyze_housing_contract(text, perspective="임차인")
    elif document_type == "employment_contract":
        result = analyze_employment_contract(text, perspective="근로자")
    else:
        result = analyze_document(text, document_type)

    print(
        f"자동 감지: {detection['label']} "
        f"(신뢰도 {detection['confidence']}%)"
    )
    summarize_result(result)
    return result

try:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        uploaded_name, uploaded_data = next(iter(uploaded.items()))
        uploaded_text = extract_text(uploaded_name, uploaded_data)
        print(f"추출된 문자 수: {len(uploaded_text):,}")
        uploaded_result = analyze_text_automatically(uploaded_text)
        print("\n[관련 공식 법률 근거 검색]")
        pprint(search_legal_basis(uploaded_text))
except ImportError:
    print("파일 업로드 셀은 Google Colab에서 사용할 수 있습니다.")

## 9. 전체 자동 테스트

저장소에 포함된 테스트를 실행하여 분석기와 Streamlit 입력 흐름을 검증합니다.

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
test_result.check_returncode()

## 제출 구성

- 웹 서비스: Streamlit 배포 URL
- 실행 노트북: `privacy_consent_review_colab.ipynb`
- 실행 ZIP: `privacy-consent-review-ai-colab.zip`
- 전체 소스 및 법률 규칙: GitHub 저장소
- AI 모델: 문서 유형, 조항 위험 유형, 계약 필드 후보, BIO 필드 값 span fallback, Transformer NER 선택 모델
- 분석 방식: 머신러닝 예측과 공식 법령 기반 규칙 엔진을 결합한 하이브리드 구조

본 결과는 법률 자문이나 위법 여부 판정이 아닙니다.